### Ingest Sprints Folder
Note: The data is stored in multi-line json

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

#### Step 1 - Ingest Data

In [0]:
# define the schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, FloatType

sprints_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName', StringType()),
    StructField('round', IntegerType()),
    StructField('season', IntegerType()),
    StructField('url', StringType()),
    StructField('constructorId', StringType()),
    StructField('driverId', StringType()),
    StructField('grid', IntegerType()),
    StructField('laps', IntegerType()),
    StructField('number', IntegerType()),
    StructField('points', FloatType()),
    StructField('position', IntegerType()),
    StructField('positionText', StringType()),
    StructField('status', StringType()),
])

In [0]:
# Read the data from Sprints file
sprints_df = (
    spark.read
        .format('json')
        .schema(sprints_schema)
        .option('mode', 'FAILFAST')
        .option('multiLine', True)
        .load(source_file)
)

#### Step 2 - Add Metadata Columns

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

#### Step 3 - Write to Bronze Delta Table

In [0]:
(
    sprints_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)